In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
from google import genai
import openai


In [2]:
df = pd.read_json('../../../data/Data_With_Details.jsonl', lines=True)
print(df.head())

                              review_id product_id  \
0  cf3980d1-5807-4936-8c5e-1bbe92e5a29e       B002   
1  544e1a92-acb6-4585-b167-51ea58b29d2b       B001   
2  7e9c3ef5-2310-421e-a83d-10a2c3dabced       B005   
3  dc92464b-85ab-46cd-86b6-9d849fe38d9f       B002   
4  26eb0926-1ff3-4b82-9435-ce11b4971551       B010   

                   product_name reviewer_id  rating     review_title  \
0  Stainless Steel Water Bottle     R719176       1  Could be better   
1              Wireless Earbuds     R331148       1  Could be better   
2             Bluetooth Speaker     R207175       1    Pretty decent   
3  Stainless Steel Water Bottle     R182627       3  Amazing quality   
4           Mechanical Keyboard     R183667       3  Could be better   

                                         review_text  verified_purchase  \
0  This product has highly recommended. could be ...              False   
1     This product has easy to use. could be better.              False   
2  This product h

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1584 entries, 0 to 1583
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   review_id          1584 non-null   object        
 1   product_id         1584 non-null   object        
 2   product_name       1584 non-null   object        
 3   reviewer_id        1584 non-null   object        
 4   rating             1584 non-null   int64         
 5   review_title       1584 non-null   object        
 6   review_text        1584 non-null   object        
 7   verified_purchase  1584 non-null   bool          
 8   helpful_votes      1584 non-null   int64         
 9   review_date        1584 non-null   object        
 10  date               1584 non-null   datetime64[ns]
 11  details            1584 non-null   object        
dtypes: bool(1), datetime64[ns](1), int64(2), object(8)
memory usage: 137.8+ KB


In [4]:
def preprocess_review(review):
    return {
        'review_id': review['review_id'],
        'product_id': review['product_id'],
        'review_text': review['review_text'],
        'rating': review['rating']
    }

In [5]:
def extract_large_review_text(review):
    review_text = review.get('review_text')

    # Keep plain text as-is.
    if isinstance(review_text, str):
        return review_text

    # If review_text is a dict, prefer the 'large' field and otherwise
    # return the longest string value we can find.
    if isinstance(review_text, dict):
        large_text = review_text.get('large')
        if isinstance(large_text, str) and large_text:
            return large_text

        string_values = [value for value in review_text.values() if isinstance(value, str)]
        if string_values:
            return max(string_values, key=len)
        return str(review_text)

    # If review_text is a list, return the longest string entry.
    if isinstance(review_text, list):
        string_values = [value for value in review_text if isinstance(value, str)]
        if string_values:
            return max(string_values, key=len)
        return ' '.join(map(str, review_text))

    return '' if review_text is None else str(review_text)


In [6]:
df.head()

,review_id,product_id,product_name,reviewer_id,rating,review_title,review_text,verified_purchase,helpful_votes,review_date,date,details
0,cf3980d1-5807-4936-8c5e-1bbe92e5a29e,B002,Stainless Steel Water Bottle,R719176,1,Could be better,This product has highly recommended. could be ...,False,2,2024-04-24,2024-04-24,"{'Date First Available': 'April 24, 2024', 'Ma..."
1,544e1a92-acb6-4585-b167-51ea58b29d2b,B001,Wireless Earbuds,R331148,1,Could be better,This product has easy to use. could be better.,False,37,2024-09-15,2024-09-15,"{'Date First Available': 'September 15, 2024',..."
2,7e9c3ef5-2310-421e-a83d-10a2c3dabced,B005,Bluetooth Speaker,R207175,1,Pretty decent,This product has good value for money. pretty ...,True,24,2024-06-23,2024-06-23,"{'Date First Available': 'June 23, 2024', 'Man..."
3,dc92464b-85ab-46cd-86b6-9d849fe38d9f,B002,Stainless Steel Water Bottle,R182627,3,Amazing quality,This product has battery life could be better....,False,40,2024-01-23,2024-01-23,"{'Date First Available': 'January 23, 2024', '..."
4,26eb0926-1ff3-4b82-9435-ce11b4971551,B010,Mechanical Keyboard,R183667,3,Could be better,This product has customer support was helpful....,True,6,2024-12-26,2024-12-26,"{'Date First Available': 'December 26, 2024', ..."


In [10]:
data_to_embed = df[['product_name', 'review_text', 'rating', 'details']].sample(n=100, random_state=42)

Define the embedding function

In [11]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: False
Qdrant API Key present: False


In [13]:
response = openai.embeddings.create(
    input = "Random Text",
    model = "text-embedding-3-small"
)

In [14]:
len(response.data[0].embedding)

1536

In [15]:
def get_embedding(text):
    response = openai.embeddings.create(
        input = text,
        model = "text-embedding-3-small"
    )
    return response.data[0].embedding

In [16]:
get_embedding("This is a sample review text to test the embedding function.")

[0.01064300537109375,
 0.033905029296875,
 0.0261383056640625,
 -0.01073455810546875,
 -0.01123046875,
 -0.038726806640625,
 0.0367431640625,
 0.022308349609375,
 0.004425048828125,
 0.00673675537109375,
 0.042633056640625,
 -0.02484130859375,
 -0.031280517578125,
 -0.007648468017578125,
 0.01288604736328125,
 0.0242919921875,
 0.0086212158203125,
 0.01482391357421875,
 0.0178680419921875,
 0.0257415771484375,
 0.022796630859375,
 -0.0273284912109375,
 -0.017486572265625,
 0.05023193359375,
 -0.01277923583984375,
 -0.0440673828125,
 0.00847625732421875,
 0.041900634765625,
 0.051055908203125,
 -0.048309326171875,
 0.034423828125,
 -0.039581298828125,
 0.0197601318359375,
 -0.0172271728515625,
 -0.04541015625,
 0.045684814453125,
 0.0122222900390625,
 0.01384735107421875,
 -0.006168365478515625,
 0.0257110595703125,
 0.00141143798828125,
 0.0222320556640625,
 -0.01373291015625,
 0.0169677734375,
 -0.03302001953125,
 0.04449462890625,
 -0.013275146484375,
 -0.029144287109375,
 -0.0092544

Create Qdrant Collection

In [17]:
qdrant_client = QdrantClient(url=os.getenv('QDRANT_URL'), api_key=os.getenv('QDRANT_API_KEY'))

In [18]:
qdrant_client.create_collection(
    collection_name="amazon_reviews",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)

True

Embed Data

In [23]:
pointstructs = PointStruct(
    id=0,
    vector=get_embedding("This is a sample review text to test the embedding function."),
    payload={
        'text': 'Kishor Paroi',
        "model": "text-embedding-3-small" 
    }
)

In [24]:
pointstructs

PointStruct(id=0, vector=[0.01065826416015625, 0.033905029296875, 0.0261383056640625, -0.0107269287109375, -0.01123046875, -0.0386962890625, 0.036773681640625, 0.0222930908203125, 0.00444793701171875, 0.006702423095703125, 0.0426025390625, -0.024810791015625, -0.03131103515625, -0.00766754150390625, 0.01290130615234375, 0.0243072509765625, 0.00864410400390625, 0.01479339599609375, 0.017852783203125, 0.0257415771484375, 0.0228118896484375, -0.0273590087890625, -0.0175018310546875, 0.050262451171875, -0.01277923583984375, -0.0440673828125, 0.00849151611328125, 0.041900634765625, 0.051025390625, -0.048309326171875, 0.034423828125, -0.03961181640625, 0.019775390625, -0.0172119140625, -0.04534912109375, 0.045684814453125, 0.01219940185546875, 0.01389312744140625, -0.00614166259765625, 0.0256500244140625, 0.0014276504516601562, 0.0222625732421875, -0.01371002197265625, 0.0169830322265625, -0.033050537109375, 0.044464111328125, -0.01323699951171875, -0.029144287109375, -0.00920867919921875, 0

Write embedded data to qdrant

In [26]:
qdrant_client.upsert(
    collection_name="amazon_reviews",
    wait=True,
    points=[pointstructs]
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

Define a function for data retrieval

In [27]:
def retrieve_data(query,k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="amazon_reviews",
        query=query_embedding,
        limit=k
    )
    return results

In [28]:
retrieve_data('What ind of charging card do you offer?', k=10).points

[ScoredPoint(id=0, version=1, score=0.14134447, payload={'text': 'Kishor Paroi', 'model': 'text-embedding-3-small'}, vector=None, shard_key=None, order_value=None)]